In [56]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [57]:
df=pd.read_csv('stud.csv')

In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


In [59]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group D,some college,standard,completed,59,70,78
1,male,group D,associate's degree,standard,none,96,93,87
2,female,group D,some college,free/reduced,none,57,76,77
3,male,group B,some college,free/reduced,none,70,70,63
4,female,group D,associate's degree,standard,none,83,85,86


In [60]:
df.columns

Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'test preparation course', 'math score', 'reading score',
       'writing score'],
      dtype='str')

In [61]:
print("Categories is 'gender' variable:  ",end=" ")
print(df['gender'].unique())
print("Categories in 'race/ethinicity' variable: ",end=" ")
print(df['race/ethnicity'].unique())
print("Categories in 'parental level of education' variable:",end="")
print(df['parental level of education'].unique())
print("Categories in lunch variable:  ",end=" ")
print(df['lunch'].unique())
print("Categories in 'test prepation course' variable: ",end=" ")
print(df['test preparation course'].unique())

Categories is 'gender' variable:   <StringArray>
['female', 'male']
Length: 2, dtype: str
Categories in 'race/ethinicity' variable:  <StringArray>
['group D', 'group B', 'group C', 'group E', 'group A']
Length: 5, dtype: str
Categories in 'parental level of education' variable:<StringArray>
[      'some college', 'associate's degree',   'some high school',
  'bachelor's degree',    'master's degree',        'high school']
Length: 6, dtype: str
Categories in lunch variable:   <StringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
Categories in 'test prepation course' variable:  <StringArray>
['completed', 'none']
Length: 2, dtype: str


In [62]:
X=df.drop(columns='math score')

In [63]:
y=df['math score']

In [64]:
print(X.head())

   gender race/ethnicity parental level of education         lunch  \
0  female        group D                some college      standard   
1    male        group D          associate's degree      standard   
2  female        group D                some college  free/reduced   
3    male        group B                some college  free/reduced   
4  female        group D          associate's degree      standard   

  test preparation course  reading score  writing score  
0               completed             70             78  
1                    none             93             87  
2                    none             76             77  
3                    none             70             63  
4                    none             85             86  


In [65]:
num_features=X.select_dtypes(exclude="str").columns
cat_features=X.select_dtypes(include="str").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_transformer=OneHotEncoder()

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer,cat_features),
        ("StandardScaler", numeric_transformer, num_features),  
    ]
)

In [66]:
X=preprocessor.fit_transform(X)

In [67]:
X

array([[ 1.        ,  0.        ,  0.        , ...,  0.        ,
        -0.02709151,  0.58994292],
       [ 0.        ,  1.        ,  0.        , ...,  1.        ,
         1.60407283,  1.18920774],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.39842962,  0.52335794],
       ...,
       [ 1.        ,  0.        ,  1.        , ...,  0.        ,
         1.10763151,  1.12262276],
       [ 0.        ,  1.        ,  0.        , ...,  1.        ,
         0.11474887, -0.47541676],
       [ 0.        ,  1.        ,  0.        , ...,  1.        ,
        -1.65825585, -1.60736142]], shape=(1000, 19))

In [68]:
from sklearn.model_selection import train_test_split

In [69]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [70]:
X_train.shape,X_test.shape

((800, 19), (200, 19))

Creat A EValuate function

In [75]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, predicted)

    return mae, rmse, r2

In [82]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
models={
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoost Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor":AdaBoostRegressor()
}
model_list=[]
r2_list=[]

for name, model in models.items():
    
    model.fit(X_train,y_train) # Train model

    # Make predictions

    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    # Evaluate Train and Test dataset

    model_train_mae, model_train_rmse, model_train_r2=evaluate_model(y_train, y_train_pred)

    model_test_mae, model_test_rmse, model_test_r2=evaluate_model(y_test,y_test_pred)


    print("Model Performance training set")
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error:{:.4f}".format(model_test_mae))
    print("-R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)

    print(name)
    model_list.append(name)
    r2_list.append(model_test_r2)

Model Performance training set
- Root Mean Squared Error: 5.4720
- Mean Absolute Error:4.0735
-R2 Score: 0.8875
Linear Regression
Model Performance training set
- Root Mean Squared Error: 6.7208
- Mean Absolute Error:5.3757
-R2 Score: 0.7900
Lasso
Model Performance training set
- Root Mean Squared Error: 5.4721
- Mean Absolute Error:4.0732
-R2 Score: 0.8873
Ridge
Model Performance training set
- Root Mean Squared Error: 5.7228
- Mean Absolute Error:5.1510
-R2 Score: 0.8102
K-Neighbors Regressor
Model Performance training set
- Root Mean Squared Error: 0.0000
- Mean Absolute Error:5.8650
-R2 Score: 0.7504
Decision Tree
Model Performance training set
- Root Mean Squared Error: 2.3340
- Mean Absolute Error:4.5587
-R2 Score: 0.8535
Random Forest Regressor
Model Performance training set
- Root Mean Squared Error: 0.8825
- Mean Absolute Error:5.0672
-R2 Score: 0.8261
XGBRegressor
Model Performance training set
- Root Mean Squared Error: 3.1545
- Mean Absolute Error:4.1443
-R2 Score: 0.8745
C